In [0]:
%python
%pip install kaggle

In [0]:
%python
# Restart Python to load the newly installed kaggle package
dbutils.library.restartPython()

In [0]:
%python

# Secret scopes must be created using Databricks CLI or UI, not via dbutils
# Run this command in your terminal:
# databricks secrets create-scope kaggle-scope

print("Note: Create the 'kaggle-scope' secret scope using the Databricks CLI:")
print("  databricks secrets create-scope kaggle-scope")


In [0]:
%python

username = dbutils.secrets.get("kaggle-scope", "kaggle-username")
key = dbutils.secrets.get("kaggle-scope", "kaggle-key")

In [0]:
%python

import os

# Set Kaggle credentials as environment variables
# (alternative to ~/.kaggle/kaggle.json which is not writable on serverless)
# Strip null bytes that may be present in the secret values
os.environ['KAGGLE_USERNAME'] = username.replace('\x00', '')
os.environ['KAGGLE_KEY'] = key.replace('\x00', '')

In [0]:
%python

# STEP 4: Download dataset using Kaggle Python API
# This downloads ALL Olist files
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

print("Downloading Olist Brazilian E-Commerce dataset...")
api.dataset_download_files('olistbr/brazilian-ecommerce', path='/tmp/olist', unzip=False)
print("Download complete!")

In [0]:
%python

# STEP 5: Unzip dataset
import subprocess
result = subprocess.run(['unzip', '-o', '/tmp/olist/brazilian-ecommerce.zip', '-d', '/tmp/olist'], 
                        capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)

In [0]:
%python

# STEP 6: Upload to Unity Catalog Volume
import os
import shutil

# Create volume if it doesn't exist
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.olist_data")

volume_path = "/Volumes/workspace/default/olist_data"
workspace_temp = "/Workspace/Users/motloungmiya@gmail.com/.tmp_olist"

# Copy files from /tmp to /Workspace (serverless allows /Workspace access)
os.makedirs(workspace_temp, exist_ok=True)
for file in os.listdir("/tmp/olist"):
    if file.endswith(".csv"):
        shutil.copy2(f"/tmp/olist/{file}", f"{workspace_temp}/{file}")

# Now read from /Workspace and write to UC Volume
for file in os.listdir(workspace_temp):
    if file.endswith(".csv"):
        source = f"{workspace_temp}/{file}"
        dest = f"{volume_path}/{file}"
        df = spark.read.csv(source, header=True, inferSchema=True)
        df.write.mode("overwrite").csv(dest, header=True)
        print(f"Uploaded {file}")

print(f"All CSV files uploaded to {volume_path}")

In [0]:
%python

df = spark.read.csv(
    "/Volumes/workspace/default/olist_data/olist_customers_dataset.csv",
    header=True,
    inferSchema=True
)

df.show()
